In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../cms_hospital.db")
paradox_hospitals = pd.read_csv('../outputs/paradox_hospitals.csv')
paradox_hospitals['facility_id'] = paradox_hospitals['facility_id'].astype(str).str.zfill(6)


#### Layer 2 — Profile the inefficient hospitals
*Who are they? Do they cluster by state, ownership type (for-profit vs. non-profit vs. government), or hospital size?*

In Layer 1 we identified **99 hospitals** that spend above the national average
yet consistently underperform on clinical outcomes across CMS benchmarks.

These hospitals are not outliers in terms of resources — they have the money.
The question is why that money is not translating into quality care.

Before we can answer *why*, we need to answer *who*.

**Are these hospitals concentrated in certain states?**

**Do they share an ownership type — for-profit, non-profit, government?**

**Is there a pattern in hospital type - acute care, critical access, specialty?**


#### The Case Mix Counter-Argument

A legitimate question before profiling these hospitals is:
*"Do they spend more because they treat more complex, sicker patients?"*

This is known as **case mix** in health economics — the idea that hospitals
treating higher-risk patients will naturally show higher spending AND worse
outcomes, not because of poor quality, but because of patient complexity.

Our dataset does not contain patient volume, diagnosis mix, or severity scores,
so we cannot fully rule this out.

However, two things weaken this counter-argument:

1. **CMS already risk-adjusts its benchmark comparisons.** The `_compared` flags
   used to build our outcome score account for expected performance given patient
   population — a hospital is only flagged "Worse than national rate" if it
   underperforms *relative to what is expected for its patients.*

2. **Hospital type is in our data.** Teaching hospitals and major medical centers
   treat the most complex patients by design — if case mix were the explanation,
   we would expect to see those hospital types dominating our 99.
   That is exactly what Layer 2 will test.

If the 99 paradox hospitals are concentrated in a specific ownership type or
geographic region rather than in teaching/complex-care hospitals, the case mix
argument weakens further — and the systemic explanation becomes much harder to ignore.

In [11]:

state_concentration = pd.read_sql_query("""
    WITH all_state_counts AS (
        SELECT state, COUNT(*) AS total_hospitals
        FROM hospital_general_info
        GROUP BY state
    ),
    paradox_state_counts AS (
        SELECT h.state, COUNT(*) AS paradox_count
        FROM hospital_general_info h
        WHERE h.facility_id IN ({})
        GROUP BY h.state
    )
    SELECT
        a.state,
        
        p.paradox_count,
        ROUND(p.paradox_count * 100.0 / a.total_hospitals, 1) AS pct_of_state_hospitals
    FROM all_state_counts a
    INNER JOIN paradox_state_counts p ON a.state = p.state
    ORDER BY paradox_count DESC
""".format(','.join(['?']*len(paradox_hospitals))), conn,
params=paradox_hospitals['facility_id'].tolist())

state_concentration

,state,paradox_count,pct_of_state_hospitals
0,CA,15,4.0
1,KY,7,6.9
2,NY,7,3.7
3,TX,7,1.5
4,AR,6,6.7
5,IL,6,3.1
6,MO,4,3.3
7,MS,4,3.8
8,NJ,4,5.1
9,FL,3,1.4


## Finding — State Concentration of Paradox Hospitals

The 99 paradox hospitals are spread across 24 states.
Raw counts are misleading — California has the most (15) but runs
a massive hospital system, making it less alarming proportionally.

The real concentration is in smaller states:

| State | Total Hospitals | Paradox Count | % of State System |
|-------|----------------|---------------|-------------------|
| KY    | 102            | 7             | 6.9%              |
| AR    | ~89            | 6             | 6.7%              |
| NJ    | 79             | 4             | 5.1%              |
| MS    | 106            | 4             | 3.8%              |
| MO    | 121            | 4             | 3.3%              |

**Key insight:** California's 15 paradox hospitals represent only 4.0%
of its system. Kentucky and Arkansas each have fewer paradox hospitals
in raw numbers — but nearly 1 in 15 of their hospitals is in the paradox zone.

The paradox is disproportionately concentrated in smaller southern
and midwestern states. These are not high-volume medical hubs
overwhelmed by complex patients. That makes the case mix
counter-argument even weaker for these states specifically —
and points more firmly toward structural and operational explanations.

## Finding — Small States, Big Problem

States like West Virginia, Mississippi, New Jersey, and Missouri
are not major medical hubs. They do not run the largest hospital systems
and they are not known for treating the most complex, high-risk patients.

Yet they appear disproportionately in our paradox group.

This is significant because it directly weakens the case mix counter-argument:

> *"If patient complexity were driving the high spending and poor outcomes,
> we would expect to see the paradox concentrated in large urban systems —
> New York, Chicago, Houston — where the sickest patients are treated."*

That is not what the data shows.

The paradox is not just a big-city, high-volume problem.
It is embedded in smaller state systems where patient complexity
is a less convincing explanation.

This shifts the question from **who these hospitals treat**
to **how these hospitals operate** — and that is exactly
what the next layers of this analysis will investigate.

In [12]:
ownership_breakdown = pd.read_sql_query("""
    WITH all_ownership_counts AS (
        SELECT hospital_ownership, COUNT(*) AS total_hospitals
        FROM hospital_general_info
        GROUP BY hospital_ownership
    ),
    paradox_ownership_counts AS (
        SELECT h.hospital_ownership, COUNT(*) AS paradox_count
        FROM hospital_general_info h
        WHERE h.facility_id IN ({})
        GROUP BY h.hospital_ownership
    )
    SELECT
        a.hospital_ownership,
        COALESCE(p.paradox_count, 0) AS paradox_count,
        ROUND(COALESCE(p.paradox_count, 0) * 100.0 / a.total_hospitals, 1) AS pct_of_ownership_type
    FROM all_ownership_counts a
    LEFT JOIN paradox_ownership_counts p ON a.hospital_ownership = p.hospital_ownership
    WHERE p.paradox_count > 0
    ORDER BY paradox_count DESC
""".format(','.join(['?']*len(paradox_hospitals))), conn,
params=paradox_hospitals['facility_id'].tolist())

ownership_breakdown

,hospital_ownership,paradox_count,pct_of_ownership_type
0,Voluntary non-profit - Private,45,2.0
1,Proprietary,19,1.8
2,Government - Hospital District or Authority,12,2.3
3,Voluntary non-profit - Other,7,2.0
4,Government - Local,6,1.5
5,Voluntary non-profit - Church,6,2.2
6,Government - State,4,1.9


#### Finding — Ownership Type of Paradox Hospitals

The 99 paradox hospitals are distributed across all ownership types.
No single type dominates — rates range from 1.5% to 2.3%.


**The headline finding:** Non-profit hospitals account for
**58 out of 99 paradox hospitals — 58% of the paradox group.**

This is counterintuitive. The common assumption is that for-profit hospitals,
driven by profit over care, would dominate this list.
They don't. For-profit hospitals actually have the lowest rate at 1.8%.

The paradox is not an ownership problem.
It cuts across all types — non-profit, for-profit, and government alike.
This means the explanation is not structural ownership —
it lies somewhere else. That is what the next layers will investigate.

In [13]:
hospital_type_breakdown = pd.read_sql_query("""
    WITH all_type_counts AS (
        SELECT hospital_type, COUNT(*) AS total_hospitals
        FROM hospital_general_info
        GROUP BY hospital_type
    ),
    paradox_type_counts AS (
        SELECT h.hospital_type, COUNT(*) AS paradox_count
        FROM hospital_general_info h
        WHERE h.facility_id IN ({})
        GROUP BY h.hospital_type
    )
    SELECT
        a.hospital_type,
        a.total_hospitals,
        p.paradox_count,
        ROUND(p.paradox_count * 100.0 / a.total_hospitals, 1) AS pct_of_type
    FROM all_type_counts a
    INNER JOIN paradox_type_counts p ON a.hospital_type = p.hospital_type
    ORDER BY paradox_count DESC
""".format(','.join(['?']*len(paradox_hospitals))), conn,
params=paradox_hospitals['facility_id'].tolist())

hospital_type_breakdown

,hospital_type,total_hospitals,paradox_count,pct_of_type
0,Acute Care Hospitals,3116,99,3.2


## Finding — Hospital Type: 100% Acute Care

Every single one of the 99 paradox hospitals is an **Acute Care Hospital.**
Not one Critical Access, specialty, or other hospital type appears in the group.

| Hospital Type         | Total Hospitals | Paradox Count | % of Type |
|-----------------------|----------------|---------------|-----------|
| Acute Care Hospitals  | 3,116          | 99            | 3.2%      |

**Why this matters:**

Acute Care Hospitals are the most resourced hospital type in the system.
They handle surgeries, emergencies, and complex procedures.
They have the largest budgets, the most staff, and the most technology.

This finding closes the door on one remaining version of the case mix argument:

> *"These hospitals spend more because they are small and struggling."*

They are not small. They are not struggling for resources.
They are fully equipped facilities with everything needed to perform well —
and 3.2% of them are still spending above average while delivering
below average clinical outcomes.

The problem is not a lack of resources.
The next layers of this analysis w

In [14]:
paradox_profile = pd.read_sql_query("""
    SELECT
        h.facility_id,
        h.facility_name,
        h.state,
        h.hospital_type,
        h.hospital_ownership,
        COUNT(*) OVER (PARTITION BY h.state) AS paradox_count_by_state,
        COUNT(*) OVER (PARTITION BY h.hospital_ownership) AS paradox_count_by_ownership,
        COUNT(*) OVER (PARTITION BY h.hospital_type) AS paradox_count_by_type
    FROM hospital_general_info h
    WHERE h.facility_id IN ({})
    ORDER BY h.state
""".format(','.join(['?']*len(paradox_hospitals))), conn,
params=paradox_hospitals['facility_id'].tolist())

print(f"Total: {len(paradox_profile)} hospitals")
paradox_profile

Total: 99 hospitals


,facility_id,facility_name,state,hospital_type,hospital_ownership,paradox_count_by_state,paradox_count_by_ownership,paradox_count_by_type
0,010023,BAPTIST MEDICAL CENTER SOUTH,AL,Acute Care Hospitals,Government - Hospital District or Authority,2,12,99
1,010039,HUNTSVILLE HOSPITAL,AL,Acute Care Hospitals,Government - Hospital District or Authority,2,12,99
2,040016,University of Arkansas Medical Sciences,AR,Acute Care Hospitals,Government - State,6,4,99
3,040078,NATIONAL PARK MEDICAL CENTER,AR,Acute Care Hospitals,Proprietary,6,19,99
4,040020,ST BERNARDS MEDICAL CENTER,AR,Acute Care Hospitals,Voluntary non-profit - Private,6,45,99
...,...,...,...,...,...,...,...,...
94,450162,GRACE SURGICAL HOSPITAL,TX,Acute Care Hospitals,Voluntary non-profit - Private,7,45,99
95,500088,VALLEY MEDICAL CENTER,WA,Acute Care Hospitals,Government - Hospital District or Authority,2,12,99
96,500064,HARBORVIEW MEDICAL CENTER,WA,Acute Care Hospitals,Government - Local,2,6,99
97,510046,PRINCETON COMMUNITY HOSPITAL ASSN INC,WV,Acute Care Hospitals,Government - Local,2,6,99


In [15]:
paradox_profile.to_csv('../outputs/layer2_paradox_profile.csv', index=False)
print(f"Saved {len(paradox_profile)} hospitals to outputs/layer2_paradox_profile.csv")

Saved 99 hospitals to outputs/layer2_paradox_profile.csv
